# 14 · Qualitative applications on OOD images (Phase 5 R9 / Phase 7 tasks ii–iii)

**T4 GPU runtime, ~20–30 min.** Runs the official application scripts on a few Set A images:
- **Retargeting** to a 1:2 (wide) and 2:1 (tall) canvas: how does structure survive a change of shape?
- **Tileable generation** on the patterns: is the periodicity preserved across the wrap-around?

Uses the fused-attention (`exact_flash_attn`) presets, which never build the N×N matrix, so the large targets fit the T4.
Outputs go to `MyDrive/CS787/outputs/qualitative/` and a contact sheet is shown below (qualitative only, no metrics).

In [ ]:
# ==== SETTINGS (edit once) ====================================================================
CODE_SOURCE   = "github"                                      # "github" or "drive"
GITHUB_REPO   = "https://github.com/rishabh-vr/CS787-Single-Image-Diffusion-Editor.git"
GITHUB_BRANCH = "main"
REPO_SUBDIR   = ""                                            # project files are at the repo root
DRIVE_CODE    = "/content/drive/MyDrive/CS787/code/Project"   # used when CODE_SOURCE == "drive"
DRIVE_ROOT    = "/content/drive/MyDrive/CS787"                # outputs, checkpoints, tables, env logs
SID_COMMIT    = "85381e3c6f29fc07144da854d86be07150404716"     # official code, audited commit
SINDDM_COMMIT = "e7107d9fd198928f26e278f99ec428e261726a45"  # SinDDM, audited commit
# If the GitHub repo is private: add a Colab secret named GH_TOKEN (key icon in the left bar)
# holding a fine-grained, read-only token for that repo.

NOTEBOOK = "14_applications"
RETARGET = ["fp_house2", "fp_layout", "pat_p4", "pat_cmm", "ckt_24aa", "dia_life"]
TILEABLE = ["pat_p4", "pat_cmm", "pat_p2", "pat_demi1"]

In [ ]:
import torch, subprocess
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU (then rerun)."
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                     capture_output=True, text=True).stdout)
print("torch", torch.__version__, "| CUDA", torch.version.cuda,
      "| bf16 native:", torch.cuda.get_device_capability(0) >= (8, 0))

In [ ]:
import os, shutil, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
for sub in ("outputs", "checkpoints", "tables", "env_logs", "data"):
    os.makedirs(f"{DRIVE_ROOT}/{sub}", exist_ok=True)

PROJECT = "/content/project"
if os.path.exists(PROJECT):
    shutil.rmtree(PROJECT)
if CODE_SOURCE == "github":
    url = GITHUB_REPO
    try:
        from google.colab import userdata
        token = userdata.get("GH_TOKEN")
        url = GITHUB_REPO.replace("https://", f"https://x-access-token:{token}@")
    except Exception:
        pass  # public repo: no token needed
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", GITHUB_BRANCH, url, "/content/repo"], check=True)
    shutil.copytree(os.path.join("/content/repo", REPO_SUBDIR), PROJECT,
                    ignore=shutil.ignore_patterns(".venv", "external", "outputs*"))
else:
    shutil.copytree(DRIVE_CODE, PROJECT, ignore=shutil.ignore_patterns(".venv", "external", "outputs*"))
print("project code at", PROJECT, "->", sorted(os.listdir(PROJECT)))

In [ ]:
SID_ROOT = "/content/efficient-SID-code"
if not os.path.exists(SID_ROOT):
    subprocess.run(["git", "clone", "-q", "https://github.com/haojunqiu/efficient-SID-code", SID_ROOT], check=True)
subprocess.run(["git", "-C", SID_ROOT, "checkout", "-q", SID_COMMIT], check=True)
!pip install -q -r {PROJECT}/env/requirements-colab.txt
!pip install -q --no-deps -e {SID_ROOT}
from PIL import Image
OUT = f"{DRIVE_ROOT}/outputs/qualitative"; os.makedirs(OUT, exist_ok=True)
IMG = lambda i: f"{PROJECT}/data/ood/setA/{i}.png"

## Retargeting: 1:2 (wide) and 2:1 (tall)

In [ ]:
FP32 = "precision.dtype=float32 image_denoiser.patch_denoiser.bottleneck_dtype=float32"
for i in RETARGET:
    W, H = Image.open(IMG(i)).size
    for tag, size in (("wide", f"({H},{2 * W})"), ("tall", f"({2 * H},{W})")):
        dst = f"{OUT}/retarget_{i}_{tag}.png"
        if os.path.exists(dst):
            continue
        cmd = (f"cd {SID_ROOT} && python scripts/sample_retarget.py --config configs/retarget/pixel_exact_flash_attn.yaml "
               f"image_path={IMG(i)} output_size='{size}' final_output_path={dst} {FP32} image_denoiser.patch_denoiser.query_chunks=4 "
               f"2>&1 | tail -1")
        print(i, tag, size)
        !{cmd}

## Tileable generation (then tiled 2×2 to show the seams)

In [ ]:
for i in TILEABLE:
    dst = f"{OUT}/tileable_{i}.png"
    if not os.path.exists(dst):
        cmd = (f"cd {SID_ROOT} && python scripts/sample_tileable.py --config configs/tileable/pixel_exact_flash_attn.yaml "
               f"image_path={IMG(i)} final_output_path={dst} {FP32} 2>&1 | tail -1")
        print(i)
        !{cmd}
    im = Image.open(dst).convert("RGB")
    t = Image.new("RGB", (2 * im.width, 2 * im.height))
    for a in range(2):
        for b in range(2):
            t.paste(im, (a * im.width, b * im.height))
    t.save(f"{OUT}/tileable_{i}_2x2.png")

In [ ]:
import glob
from IPython.display import display
for f in sorted(glob.glob(f"{OUT}/retarget_*.png")) + sorted(glob.glob(f"{OUT}/tileable_*_2x2.png")):
    print(os.path.basename(f)); display(Image.open(f).convert("RGB").copy().resize((min(600, Image.open(f).width), int(Image.open(f).height * min(600, Image.open(f).width) / Image.open(f).width))))

In [ ]:
import datetime
stamp = datetime.datetime.now().strftime("%Y-%m-%d_%H%M")
!pip freeze > {DRIVE_ROOT}/env_logs/{stamp}_{NOTEBOOK}_freeze.txt
print("saved", f"{DRIVE_ROOT}/env_logs/{stamp}_{NOTEBOOK}_freeze.txt")